# Silver Metadata Helpers

Metadata registry and helper functions for the generic Bronze-to-Silver runtime.


In [ ]:
SILVER_TABLE_CONFIGS = {
    "orders": {
        "table_id": "orders",
        "bronze_table": "orders",
        "silver_table": "silver_orders",
        "cdc_contract_key": "cdc.orders",
        "silver_contract_key": "silver.orders",
        "checkpoint_name": "orders_silver_generic",
        "primary_keys": ["id"],
        "dedupe_order_columns": ["event_time", "event_ts_ms", "bronze_offset"],
        "merge_core_fields": ["id", "product_id", "product_legacy", "price", "created_at"],
        "bootstrap_hook": "reconcile_orders_legacy",
        "field_mappings": [
            {"target": "id", "source_paths": ["after.id", "before.id"]},
            {"target": "product_id", "source_paths": ["after.product_id", "before.product_id"]},
            {"target": "product_legacy", "source_paths": ["after.product_name", "before.product_name"]},
            {"target": "price", "transform": "decimal_from_json_paths", "json_value_paths": ["$.payload.after.price.value", "$.payload.before.price.value", "$.payload.after.price", "$.payload.before.price"], "json_scale_paths": ["$.payload.after.price.scale", "$.payload.before.price.scale"], "default_scale": 2, "precision": 12, "scale": 2},
            {"target": "created_at", "transform": "epoch_micros_to_timestamp", "source_paths": ["after.created_at", "before.created_at"]}
        ]
    },
    "products": {
        "table_id": "products",
        "bronze_table": "products",
        "silver_table": "silver_products",
        "cdc_contract_key": "cdc.products",
        "silver_contract_key": "silver.products",
        "checkpoint_name": "products_silver_generic",
        "primary_keys": ["id"],
        "dedupe_order_columns": ["event_time", "event_ts_ms", "bronze_offset"],
        "merge_core_fields": ["id", "product_name", "weight", "color", "created_at", "updated_at"],
        "bootstrap_hook": None,
        "field_mappings": [
            {"target": "id", "source_paths": ["after.id", "before.id"]},
            {"target": "product_name", "source_paths": ["after.product_name", "before.product_name"]},
            {"target": "weight", "transform": "decimal_from_debezium_bytes", "source_paths": ["after.weight", "before.weight"], "precision": 10, "scale": 2},
            {"target": "color", "source_paths": ["after.color", "before.color"]},
            {"target": "created_at", "transform": "epoch_micros_to_timestamp", "source_paths": ["after.created_at", "before.created_at"]},
            {"target": "updated_at", "transform": "epoch_micros_to_timestamp", "source_paths": ["after.updated_at", "before.updated_at"]}
        ]
    }
}


In [ ]:
def list_enabled_silver_tables() -> list[str]:
    return sorted(SILVER_TABLE_CONFIGS.keys())


def get_silver_table_config(table_id: str) -> dict:
    if table_id not in SILVER_TABLE_CONFIGS:
        available = ', '.join(list_enabled_silver_tables())
        raise ValueError(f"Unknown TABLE_ID '{table_id}'. Available: {available}")
    return SILVER_TABLE_CONFIGS[table_id]


In [ ]:
def ensure_table_from_contract(spark, table_fqn: str, schema) -> None:
    if spark.catalog.tableExists(table_fqn):
        print(f"Using existing table: {table_fqn}")
        return

    empty_df = spark.createDataFrame([], schema)
    (
        empty_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_fqn)
    )
    print(f"Created table from contract: {table_fqn}")


In [ ]:
def run_silver_bootstrap_hook(spark, table_config: dict, silver_table_fqn: str, catalog: str, silver_schema: str) -> None:
    hook_name = table_config.get("bootstrap_hook")
    if not hook_name:
        return

    if hook_name == "reconcile_orders_legacy":
        products_table_fqn = f"{catalog}.{silver_schema}.silver_products"
        reconcile_silver_orders_table(spark, silver_table_fqn, products_table_fqn)
        return

    raise ValueError(f"Unsupported bootstrap hook: {hook_name}")
